In [ ]:
import pandas as pd

import cloudscraper
from bs4 import BeautifulSoup
from datetime import datetime as dt
import traceback
import numpy as np

from config import filename

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.DataFrame(columns=['title', 'platform', "price", "psd_URL", "icon_URL", "status"])

compare_old = False 

In [ ]:
if compare_old:
    try:
        df_previous = pd.read_csv(filename)
    except FileNotFoundError:
        print("file not found")
        compare_old = False

In [ ]:
time = dt.now()

page_num = 1
df = pd.DataFrame(columns=['title', 'platform', "psd_URL", "icon_URL", "price", "status"])

prev_page = None

scraper = cloudscraper.create_scraper()

while True:
    try:

        response = scraper.get(f"https://psdeals.net/ua-store/all-games/{page_num}")

        soup = BeautifulSoup(response.text, "html.parser")

        grid_items = soup.find_all('div', class_='game-collection-item col-md-2 col-sm-4 col-xs-6')
        
        if grid_items == prev_page:
            break

        for item in grid_items:
            
            title = item.find('span', class_='game-collection-item-details-title').text
            platform = item.find('span', class_='game-collection-item-top-platform').text
            image = item.find('img', class_='game-collection-item-image lazy').get('data-src')
            psd_link = "https://psdeals.net" + item.find('a', class_='game-collection-item-link').get('href')
            
            try:
                price = item.find('span', class_='game-collection-item-price').text
            except:
                price = np.NaN
            
            new_row = {"title": title, 
                       "platform": platform,
                       "price": price,
                       "psd_URL": psd_link,
                       "icon_URL": image}

            df = df.append(new_row, ignore_index=True)

        print(f"page {page_num}  time {dt.now() - time}")
        
        page_num += 1
        
        prev_page = grid_items
        
    except Exception as e:
        print(e)
        print(traceback.format_exc())
        break

In [ ]:
df.head()

In [ ]:
delete_lst = ['PS3', 'PS Vita', 'PS3 / PS Vita / PSP', 
              'PS Vita / PSP', 'PS3 / PS Vita', 'PS3 / PSP', 'PSP' ]

for del_pl in delete_lst:
    df = df[df.platform != del_pl]

In [ ]:
if compare_old:
    df_res = pd.merge(df, df_previous, on=['title', 'platform', 'psd_URL'], how='outer')
    df_res['status'] = df_res[['status_x', 'status_y']].max(axis=1).fillna(0) 
    df_res['price'] = df_res['price_x'].combine_first(df_res['price_y'])
    #df_res['psd_URL'] = df_res['psd_URL_x'].combine_first(df_res['psd_URL_y'])
    df_res['icon_URL'] = df_res['icon_URL_x'].combine_first(df_res['icon_URL_y'])
    df_res.drop(['status_x', 'status_y', 
                  'price_x', 'price_y', 
                  'icon_URL_y', 'icon_URL_x'], inplace=True, axis=1)
    
    df_res = df_res[['title', 'platform', 'psd_URL', 'icon_URL', 'price', 'status']]
else:
    df_res = df
    df_res['status'] = 0
    
df_res = df_res.drop_duplicates(subset=['title', 'platform', 'psd_URL'], keep=False)
df_res = df_res.sort_values(by=['status', 'title'], ascending=[False, True]).reset_index(drop=True)

In [ ]:
df_res.head(5)

In [ ]:
df_res.shape

In [ ]:
try:
    df_previous.shape
except Exception:
    pass

In [ ]:
df_res.to_csv(filename, index=False)

In [ ]:
import numpy as np

In [ ]:
file_finished = "full_list_v3.csv"
file_new_empty = "full_list_v3_temp.csv"

In [ ]:
df_finished = pd.read_csv(file_finished)
df_new = pd.read_csv(file_new_empty)

In [ ]:
df_new['status'] = np.nan

In [ ]:
df_res = pd.merge(df_new, df_finished, on=['title', 'platform', 'psd_URL'], how='outer')

df_res['status'] = df_res[['status_x', 'status_y']].max(axis=1).fillna(0) 
df_res['price'] = df_res['price_x'].combine_first(df_res['price_y'])
#df_res['psd_URL'] = df_res['psd_URL_x'].combine_first(df_res['psd_URL_y'])
df_res['icon_URL'] = df_res['icon_URL_x'].combine_first(df_res['icon_URL_y'])
df_res.drop(['status_x', 'status_y', 
              'price_x', 'price_y', 
              'icon_URL_y', 'icon_URL_x'], inplace=True, axis=1)

df_res = df_res[['title', 'platform', 'psd_URL', 'icon_URL', 'price', 'status']]

df_res = df_res.drop_duplicates(subset=['title', 'platform', 'psd_URL'], keep=False)
df_res = df_res.sort_values(by=['status', 'title'], ascending=[False, True]).reset_index(drop=True)

In [ ]:
df_res.status.value_counts()

In [ ]:
df_res.to_csv(filename)